In [2]:
import os
import json
import math
import itertools
import hashlib
import joblib
import numpy as np
import pandas as pd

In [3]:
from datetime import datetime, timezone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
if os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
elif os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
else:
    raise FileNotFoundError(
        "Place heart.csv or heart.csv in your notebook folder."
    )

df.columns = df.columns.str.strip()
df = df.drop_duplicates()
df = df.dropna(subset=["target"])

X = df.drop(columns=["target"]).apply(
    pd.to_numeric, errors="coerce"
)
X = X.replace([np.inf, -np.inf], np.nan)

y = pd.to_numeric(df["target"], errors="coerce")
valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if y.nunique() != 2:
    raise ValueError("The target must contain exactly two classes.")

if y.value_counts().min() < 5:
    raise ValueError("At least five samples per class are needed.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dataset loaded:", df.shape)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Dataset loaded: (302, 14)
Training samples: 241
Testing samples: 61


In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            random_state=42, n_jobs=-1
        ))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),

    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

In [6]:
parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "model__solver": ["liblinear", "lbfgs"],
        "model__class_weight": [None, "balanced"]
    },

    "Decision Tree": {
        "model__max_depth": [2, 3, 5, 8, 12, None],
        "model__criterion": ["gini", "entropy"],
        "model__min_samples_split": [2, 5, 10, 20],
        "model__min_samples_leaf": [1, 2, 4, 8]
    },

    "Random Forest": {
        "model__n_estimators": [50, 100, 150, 200],
        "model__max_depth": [3, 5, 8, 12, None],
        "model__min_samples_split": [2, 5, 10],
        "model__max_features": ["sqrt", None]
    },

    "K-Nearest Neighbors": {
        "model__n_neighbors": [3, 5, 7, 9, 11, 15],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },

    "Support Vector Machine": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

In [7]:

def rbf_kernel(A, B, length_scale=0.30):
    """Calculate the Gaussian/RBF covariance matrix."""
    differences = A[:, None, :] - B[None, :, :]
    squared_distances = np.sum(differences ** 2, axis=2)

    return np.exp(
        -squared_distances / (2 * length_scale ** 2)
    )


def gaussian_process_predict(
    X_observed, y_observed, X_candidates
):
    """Predict candidate means and uncertainty from scratch."""

    K = rbf_kernel(X_observed, X_observed)
    K += np.eye(len(X_observed)) * 1e-6

    K_star = rbf_kernel(X_observed, X_candidates)
    K_candidate = rbf_kernel(X_candidates, X_candidates)

    try:
        alpha = np.linalg.solve(K, y_observed)
        mean = K_star.T @ alpha

        solved = np.linalg.solve(K, K_star)
        variance = np.diag(K_candidate) - np.sum(
            K_star * solved, axis=0
        )

        variance = np.maximum(variance, 1e-12)
        std = np.sqrt(variance)

        return mean, std

    except np.linalg.LinAlgError:
        return (
            np.full(len(X_candidates), np.mean(y_observed)),
            np.ones(len(X_candidates))
        )


def normal_pdf(z):
    return np.exp(-0.5 * z ** 2) / np.sqrt(2 * np.pi)


def normal_cdf(z):
    """Standard normal CDF using the error function."""
    return np.array([
        0.5 * (1 + math.erf(float(value) / np.sqrt(2)))
        for value in np.asarray(z).ravel()
    ]).reshape(np.asarray(z).shape)


def expected_improvement(mean, std, best_score):
    """Calculate expected improvement for maximization."""
    improvement = mean - best_score - 0.01
    z = improvement / np.maximum(std, 1e-12)

    ei = (
        improvement * normal_cdf(z)
        + std * normal_pdf(z)
    )

    return np.maximum(ei, 0)

In [8]:
def encode_parameters(params, grid):
    """Convert mixed numeric and categorical values to numbers."""

    encoded = []

    for name, choices in grid.items():
        value = params[name]

        
        index = choices.index(value)

        if len(choices) == 1:
            encoded.append(0.0)
        else:
            encoded.append(index / (len(choices) - 1))

    return encoded

In [9]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

def evaluate_parameters(base_model, params):
    """Evaluate one setting using training data only."""

    scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        candidate_model = clone(base_model)
        candidate_model.set_params(**params)

        candidate_model.fit(X_fold_train, y_fold_train)

        predictions = candidate_model.predict(X_fold_valid)

        scores.append(
            f1_score(
                y_fold_valid,
                predictions,
                zero_division=0
            )
        )

    return float(np.mean(scores))

In [10]:
N_INITIAL = 3
N_ITERATIONS = 7
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

all_bo_results = []
best_overall_score = -1.0
best_overall_name = None
best_overall_params = None
best_overall_model = None

for model_name, base_model in models.items():

    print("\n" + "=" * 55)
    print("Optimizing:", model_name)

    grid = parameter_grids[model_name]
    parameter_names = list(grid.keys())

    
    all_combinations = [
        dict(zip(parameter_names, values))
        for values in itertools.product(
            *[grid[name] for name in parameter_names]
        )
    ]

    candidate_vectors = np.array([
        encode_parameters(params, grid)
        for params in all_combinations
    ])

    
    initial_count = min(N_INITIAL, len(all_combinations))

    selected_indices = list(
        rng.choice(
            len(all_combinations),
            size=initial_count,
            replace=False
        )
    )

    observed_X = []
    observed_scores = []
    evaluated_indices = set()

    for iteration in range(
        min(N_ITERATIONS, len(all_combinations))
    ):

        if iteration >= initial_count:
            remaining = [
                i for i in range(len(all_combinations))
                if i not in evaluated_indices
            ]

            if not remaining:
                break

            candidate_X = candidate_vectors[remaining]

            mean, std = gaussian_process_predict(
                np.array(observed_X),
                np.array(observed_scores),
                candidate_X
            )

            ei = expected_improvement(
                mean,
                std,
                max(observed_scores)
            )

            next_index = remaining[int(np.argmax(ei))]
        else:
            next_index = selected_indices[iteration]

        evaluated_indices.add(next_index)

        params = all_combinations[next_index]

        score = evaluate_parameters(base_model, params)

        observed_X.append(candidate_vectors[next_index])
        observed_scores.append(score)

        all_bo_results.append({
            "Model": model_name,
            "Iteration": iteration + 1,
            "Mean_CV_F1": score,
            "Parameters": str(params)
        })

        print(
            f"Iteration {iteration + 1}/"
            f"{min(N_ITERATIONS, len(all_combinations))} | "
            f"F1: {score:.4f} | {params}"
        )

    best_index = int(np.argmax(observed_scores))
    model_best_score = observed_scores[best_index]

    evaluated_list = list(evaluated_indices)
    best_params = all_combinations[evaluated_list[best_index]].copy()

    print("Best CV F1:", round(model_best_score, 4))
    print("Best parameters:", best_params)

    if model_best_score > best_overall_score:
        best_overall_score = model_best_score
        best_overall_name = model_name
        best_overall_params = best_params.copy()
        best_overall_model = clone(base_model)
        best_overall_model.set_params(**best_overall_params)


Optimizing: Logistic Regression
Iteration 1/7 | F1: 0.8189 | {'model__C': 10.0, 'model__solver': 'liblinear', 'model__class_weight': 'balanced'}
Iteration 2/7 | F1: 0.8420 | {'model__C': 0.01, 'model__solver': 'liblinear', 'model__class_weight': 'balanced'}
Iteration 3/7 | F1: 0.8259 | {'model__C': 10.0, 'model__solver': 'lbfgs', 'model__class_weight': None}
Iteration 4/7 | F1: 0.8384 | {'model__C': 0.1, 'model__solver': 'liblinear', 'model__class_weight': 'balanced'}
Iteration 5/7 | F1: 0.8273 | {'model__C': 1.0, 'model__solver': 'lbfgs', 'model__class_weight': None}
Iteration 6/7 | F1: 0.8189 | {'model__C': 100.0, 'model__solver': 'liblinear', 'model__class_weight': 'balanced'}
Iteration 7/7 | F1: 0.8347 | {'model__C': 0.01, 'model__solver': 'lbfgs', 'model__class_weight': None}
Best CV F1: 0.842
Best parameters: {'model__C': 0.01, 'model__solver': 'lbfgs', 'model__class_weight': None}

Optimizing: Decision Tree
Iteration 1/7 | F1: 0.7967 | {'model__max_depth': 2, 'model__criterion'

In [11]:
bo_results = pd.DataFrame(all_bo_results)
bo_results = bo_results.sort_values(
    "Mean_CV_F1", ascending=False
).reset_index(drop=True)

bo_results.to_csv(
    "manual_bayesian_optimization_results.csv",
    index=False
)

In [13]:
best_overall_model.fit(X_train, y_train)

test_predictions = best_overall_model.predict(X_test)

test_metrics = {
    "Accuracy": float(
        accuracy_score(y_test, test_predictions)
    ),
    "Precision": float(
        precision_score(
            y_test, test_predictions, zero_division=0
        )
    ),
    "Recall": float(
        recall_score(
            y_test, test_predictions, zero_division=0
        )
    ),
    "F1": float(
        f1_score(
            y_test, test_predictions, zero_division=0
        )
    )
}

print("\n" + "=" * 55)
print("BEST MODEL:", best_overall_name)
print("BEST PARAMETERS:", best_overall_params)
print("BEST CV F1:", round(best_overall_score, 4))
print("\nHELD-OUT TEST RESULTS")

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")



MODEL_VERSION = "1.2.0"

safe_name = best_overall_name.lower().replace(" ", "_")

export_dir = os.path.join(
    "model_exports",
    safe_name,
    "v" + MODEL_VERSION
)

os.makedirs(export_dir, exist_ok=True)

model_path = os.path.join(
    export_dir,
    safe_name + "_v" + MODEL_VERSION + ".joblib"
)

joblib.dump(
    best_overall_model,
    model_path,
    compress=("gzip", 3),
    protocol=5
)


def calculate_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            digest.update(chunk)

    return digest.hexdigest()


metadata = {
    "model_name": best_overall_name,
    "version": MODEL_VERSION,
    "best_parameters": best_overall_params,
    "best_cross_validation_f1": float(best_overall_score),
    "test_metrics": test_metrics,
    "target": "target",
    "features": list(X.columns),
    "optimization": "Manual Gaussian Process Bayesian Optimization",
    "acquisition_function": "Expected Improvement",
    "initial_candidates_per_model": N_INITIAL,
    "iterations_per_model": N_ITERATIONS,
    "cross_validation_folds": 3,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_file": os.path.basename(model_path),
    "sha256": calculate_sha256(model_path)
}

manifest_path = os.path.join(export_dir, "manifest.json")

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

# Verify saved model
loaded_model = joblib.load(model_path)

assert np.array_equal(
    loaded_model.predict(X_test),
    test_predictions
)

print("\nBAYESIAN OPTIMIZATION COMPLETED")
print("Model file:", os.path.abspath(model_path))
print("Metadata:", os.path.abspath(manifest_path))
print(
    "Optimization results:",
    os.path.abspath("manual_bayesian_optimization_results.csv")
)
print("Saved model verification: passed")



BEST MODEL: Random Forest
BEST PARAMETERS: {'model__n_estimators': 200, 'model__max_depth': 12, 'model__min_samples_split': 5, 'model__max_features': 'sqrt'}
BEST CV F1: 0.8569

HELD-OUT TEST RESULTS
Accuracy: 0.7705
Precision: 0.7879
Recall: 0.7879
F1: 0.7879

BAYESIAN OPTIMIZATION COMPLETED
Model file: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/random_forest/v1.2.0/random_forest_v1.2.0.joblib
Metadata: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/random_forest/v1.2.0/manifest.json
Optimization results: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/manual_bayesian_optimization_results.csv
Saved model verification: passed
